In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna 
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import missingno as msno

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline, make_pipeline, FunctionTransformer
import statsmodels.api as sm
from sklearn.feature_selection import SequentialFeatureSelector
#from utils.perm_class import ClassificationCV
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression, RidgeClassifier
pd.set_option('display.max_rows', None)

Using 10% of data (stratified & randomized) for EDA, when fitting model, yield

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

server = '.\SQLEXPRESS' 
database = 'ClubLoanData'

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    f"DRIVER={{SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

engine = create_engine(connection_url)

try:
    query = """SELECT * FROM (
    SELECT TOP 10 PERCENT * FROM dbo.loan_model_ready 
    WHERE predictor = 1 
    ORDER BY NEWID()) AS Defaults

UNION ALL

SELECT * FROM (
    SELECT TOP 10 PERCENT * FROM dbo.loan_model_ready 
    WHERE predictor = 0 
    ORDER BY NEWID()) AS GoodLoans
"""
    df = pd.read_sql(query, engine)
    df.to_csv('Data/loan_data_sample.csv')
    print(f"Data Shape: {df.shape}")
except Exception as e:
    print(f"Error: {e}")


In [2]:

df = pd.read_csv('Data\loan_data_sample.csv')
features = ['loan_amnt', 'term', 'emp_length', 'home_ownership',
       'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'mths_since_last_major_derog',
       'application_type', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
       'total_il_high_credit_limit',
       'months_sincefrst_credit', 'public_record', 'is_consolidation',
       'addr_state', 'is_currently_delinq', 'has_il_history']

index_sql = 'Loan_ID'
target = 'predictor'

df_features  = df[features]
df_predictor = pd.Series(df[target])

#print(df_features.shape,df_predictor.shape)

X_train, X_test, y_train,y_test = train_test_split(df_features,df_predictor,stratify=df_predictor,test_size=.2,random_state=11)

categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

Univariate Analysis of Numerical Col's

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
print(X_train[numerical_features].describe())

Changing the imputed '999' back to NaN for T-Test. This is to make it more straightforward for other Col's with NaN.

In [3]:
imputed_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq'
]

for col in imputed_cols:
    X_train[col] = X_train[col].replace(999.0, np.nan)

In [ ]:
pd.set_option('display.max_rows', None)

result=[]
for column in numerical_features:
    group1 = X_train[y_train == 0 ][column].dropna()
    group2 = X_train[y_train == 1 ][column].dropna()

    t_stat,p_val = stats.ttest_ind(group1,group2,equal_var=False)

    result.append({'Column':column,'p_val':p_val,
                   'Decision': 'Drop' if p_val > 0.05 else 'Keep'})

results = pd.DataFrame(result).sort_values(by='p_val')
print(results)

Univariate Analysis of Categorical Columns

In [ ]:
from scipy.stats import chi2_contingency

cat_result=[]
for column in categorical_features:
    contingency_table = pd.crosstab(X_train[column], y_train)
    chi2, p_val, dof, expected = chi2_contingency(contingency_table)

    cat_result.append({'Column':column,'p_val':p_val,
                   'Decision': 'Drop' if p_val > 0.05 else 'Keep'})

cat_results = pd.DataFrame(cat_result).sort_values(by='p_val')
print(cat_results)

Dropping the insignificant cols

In [4]:
cols_to_drop = ['mths_since_last_major_derog','total_cu_tl','num_tl_30dpd','open_act_il','chargeoff_within_12_mths','is_consolidation',
                'total_bal_ex_mort','total_bal_il','total_il_high_credit_limit','num_tl_120dpd_2m']

X_train = X_train.drop(columns=cols_to_drop)


Target Encoding / Probability Encoding. More effective for highly cardinal data like addr_state. Rest of categorical columns will be OHE

- for IA (which had a mean of 0) we'll be using Bayesian smoothing

In [8]:
categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()
print(categorical_features)

['home_ownership', 'verification_status', 'application_type', 'addr_state']


In [5]:
categorical_features = ['home_ownership', 'verification_status', 'application_type', 'addr_state']

X_temp_data = X_train.copy()
X_temp_data['predictor'] = y_train
state_means = X_temp_data.groupby('addr_state')['predictor'].mean()
global_default_mean = y_train.mean()



In [6]:
#print(X_temp_data.groupby('addr_state').count())
#print(X_temp_data.groupby('addr_state')['predictor'].sum())

m= 10 #smoothing parameter
state_means = X_temp_data.groupby('addr_state')['predictor'].mean()
state_counts = X_temp_data['addr_state'].value_counts()
means_smoothed = ((state_counts*state_means)+(m*global_default_mean))/(state_counts+m)



In [9]:
X_train['state_enc'] = X_train['addr_state'].map(means_smoothed)
X_test['state_enc'] = X_test['addr_state'].map(means_smoothed)
X_test['state_enc'] = X_test['state_enc'].fillna(global_default_mean)
categorical_features.remove('addr_state')

X_encoded=pd.get_dummies(X_train[categorical_features],drop_first=True,sparse=False,dtype=int)
X_train = pd.concat([X_train[numerical_features],X_encoded],axis=1)

X_encoded_test = pd.get_dummies(X_test[categorical_features],drop_first=True,sparse=False,dtype=int)
X_test = pd.concat([X_test[numerical_features],X_encoded_test],axis=1)
#align columns
X_test = X_test.reindex(columns=X_train.columns,fill_value=0)

Lets Diagnose our columns with NA values.

In [ ]:
missing_counts = X_train.isnull().sum()
print(missing_counts)

In [ ]:
#print(X_train['all_util'].head(15))
print(missing_counts[(missing_counts < 5000)&(missing_counts>0) ].index.tolist())

Lets create a pipeline instead

In [10]:
from sklearn import set_config


zero_cols = [
    'max_bal_bc', 'all_util', 'il_util', 'open_acc_6m', 
    'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m', 'inq_last_12m'
]

# Group 2: NaN means Never happened (Requires a Flag + Imputation)
flag_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 
    'mths_since_recent_inq',"mths_since_rcnt_il"
]

# Group 3:  missing data (Fill with Median, no flag)
median_cols = [
    'months_sincefrst_credit', 'annual_inc', 'inq_last_6mths', 
    'revol_util', 'total_acc', 'pub_rec', 'open_acc', 
    'mo_sin_old_rev_tl_op', 'num_rev_accts', 'tot_hi_cred_lim','acc_open_past_24mths',
    'num_bc_sats','num_sats','mort_acc','mths_since_recent_bc','total_bc_limit'
    ,'pub_rec_bankruptcies'
]


set_config(transform_output="pandas")

preprocessing = ColumnTransformer([
    ('zeros',SimpleImputer(strategy='constant',fill_value=0),zero_cols),
    ('flags',SimpleImputer(strategy= 'median',add_indicator=True),flag_cols),
    ('median',SimpleImputer(strategy='median'),median_cols)
],remainder='passthrough')

X_train = preprocessing.fit_transform(X_train)
X_test = preprocessing.transform(X_test)




In [11]:
missing_counts = X_train.isnull().sum()
print(missing_counts)

zeros__max_bal_bc                          0
zeros__all_util                            0
zeros__il_util                             0
zeros__open_acc_6m                         0
zeros__open_il_12m                         0
                                          ..
remainder__home_ownership_OTHER            0
remainder__home_ownership_OWN              0
remainder__home_ownership_RENT             0
remainder__verification_status_Verified    0
remainder__application_type_Joint App      0
Length: 79, dtype: int64


Correlation / VIF

In [13]:
pd.set_option('display.max_rows', None)
corr = X_train.corrwith(y_train)
corr = corr.abs().sort_values(ascending=False)

corr_df= corr.to_frame(name='Correlation')
print(corr_df)

                                                    Correlation
remainder__term                                        0.174618
remainder__dti                                         0.107527
median__acc_open_past_24mths                           0.097360
remainder__verification_status_Verified                0.089248
remainder__num_tl_op_past_12m                          0.085999
remainder__bc_open_to_buy                              0.085743
remainder__avg_cur_bal                                 0.077905
median__tot_hi_cred_lim                                0.074702
zeros__open_rv_24m                                     0.074029
median__mort_acc                                       0.073950
remainder__percent_bc_gt_75                            0.073569
remainder__bc_util                                     0.073348
remainder__num_actv_rev_tl                             0.071567
remainder__num_rev_tl_bal_gt_0                         0.070012
median__total_bc_limit                  

run vif, then we'll do sm.logit to see sig 

In [ ]:
X_train_na = X_train.dropna()

vif = pd.DataFrame()
vif['features']=X_train_na.columns

vif['VIF'] = [variance_inflation_factor(X_train_na.values, i ) 
              for i in range(len(X_train_na.columns))]
vif = vif.sort_values(by='VIF',ascending=False)

print(vif)

In [ ]:
vif_list = list(vif[vif['VIF']>10]['features'])

print(vif_list)

Correlation Heatmap - diagnosing VIF

In [ ]:
dfeatures = ['revol_bal','mths_since_recent_revol_delinq','num_op_rev_tl',
             'total_rev_hi_lim','open_il_12m','bc_open_to_buy','num_rev_accts',
             'total_acc','open_il_24m','total_bc_limit','num_rev_tl_bal_gt_0','num_actv_rev_tl',
             'home_ownership_OWN','open_rv_12m','tot_hi_cred_lim','tot_cur_bal',
             'num_tl_op_past_12m','open_rv_24m','acc_open_past_24mths','home_ownership_RENT',
             'home_ownership_MORTGAGE','num_sats','open_acc']



X_cor = X_train[dfeatures]
plt.figure(figsize=(20,10))
sns.heatmap(X_cor.corr(numeric_only=True), annot=True,fmt='.2f',cmap='vlag')
plt.title('Linear Correlation between features')

#plt.savefig('Images/corr_heatmap.png',dpi=300,bbox_inches='tight')

plt.show()

Cols to drop

In [ ]:
cols_drop = ['revol_bal','num_rev_accts','total_bc_limit','open_rv_12m','open_rv_24m','num_sats', 'num_rev_tl_bal_gt_0' , 'tot_cur_bal']

X_train_na = X_train_na.drop(columns=cols_drop)

In [ ]:

vif2 = pd.DataFrame()
vif2['features']=X_train_na.columns

vif2['VIF'] = [variance_inflation_factor(X_train_na.values, i ) 
              for i in range(len(X_train_na.columns))]
vif2 = vif2.sort_values(by='VIF',ascending=False)

print(vif2)

In [ ]:
cols_drop = ['revol_bal','num_rev_accts','total_bc_limit','open_rv_12m','open_rv_24m','num_sats', 'num_rev_tl_bal_gt_0' , 'tot_cur_bal']

X_train = X_train.drop(columns=cols_drop)

Imputing -- First we'll Impute / remove the col's with wayy too many NaN's... Not contributing much - or impute 

In [ ]:
print(X_train.shape)

In [ ]:
categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()
nan1 = X_train[numerical_features].isnull().sum()
print(nan1)

In [ ]:
zero_impute_cols = ['all_util', 'il_util', 'max_bal_bc', 'open_acc_6m', 'open_il_12m', 'open_il_24m', 'inq_last_12m']
for col in zero_impute_cols:
    X_train[col] = X_train[col].fillna(0)

Multivariate analysis

In [ ]:
model = sm.OLS(y_train, X_train.assign(const=1))
results = model.fit()
print(results.summary())

Now do some Outlier/ leverage points.